In [135]:
import numpy as np
import pandas as pd
import sympy as sp
from scipy.optimize import fsolve
from scipy.integrate import quad
from sympy import symbols,Eq,solve,cos,sin,tan,sqrt,exp,log,Integral,pi,atan2
from scipy.optimize import differential_evolution

In [136]:
p = 1.7
b = p / 2 / np.pi
theta_initial = 16 * 2 * np.pi
time_duration = 300
num_sections = 223
head_length = 341 / 100
body_length = 220 / 100
dlt = (2*27.5) /100  # 板凳交错
section_length = np.array([head_length] + [body_length] * (num_sections - 1))  # 创建一个包含每节板凳长度的数组
distance = section_length-dlt # 计算每节板凳之间的距离
number=30

# radius = 4.5

In [137]:
# 计算第九圈长度, 0.55在第八圈都不会撞, 那么0.85更不会, 只需要算此时开始向外围一圈即可--计算30个
def arclength(theta):
    return np.sqrt(b**2 + (b * theta)**2)
# 计算长度
length1, _ = quad(arclength, 0, 16*np.pi)
length2, _ = quad(arclength, 0, 18*np.pi)
print(length2 - length1)

90.80796017347478


In [138]:
def equations(vars, angle_prev, r_prev, dist):
    angle_i, r = vars
    eq1 = dist - np.sqrt(r**2 + r_prev**2 - 2 * r_prev * r * np.cos(angle_prev - angle_i))
    eq2 = r - b * angle_i
    return [eq1, eq2]

def eqinner(vars, a, b, dist, radius):
    x, y = vars
    d = np.sqrt(a**2 + (b-radius/3)**2)
    eq1 = x**2 + (y-radius/3*2) ** 2 - (radius/3)**2
    eq2 = x**2 + (y+radius/3) ** 2 - (radius/3*2)**2
    eq = np.sqrt((x-a)**2 + (y-b)**2) - dist
    if d < dist:
        return [eq1, eq]
    else:
        return [eq2, eq]

In [139]:
# 根据龙头位置theta, 计算向外一圈的点位的函数
def body_state(theta, r, radius):
    """得到确定龙头theta位置状态下, 整条龙各节的位置"""
    thetas = np.zeros(number+1)
    rs = np.zeros(number+1)
    thetas[0] = theta
    rs[0] = r
    x = np.zeros(number+1)
    y = np.zeros(number+1)
    x[0], y[0] = polar_to_cartesian(rs[0], thetas[0])
        
    for i in range(1, number+1):
        initial_guess = [thetas[i-1] + 0.5, rs[i-1]+0.1]  # 微调初始角度
        sol = fsolve(equations, initial_guess, args=(thetas[i-1], rs[i-1], distance[i-1]))
        thetas[i] = sol[0]       # 得到龙身点(r, theta)的theta
        rs[i] = thetas[i] * b    # 得到龙身点坐标的r
        # r, theta如果不合适,没有更新, 不是最终值
        
        if (rs[i] < radius):
            sol = fsolve(eqinner, initial_guess, args = (x[i-1], y[i-1], distance[i-1], radius))
            x[i], y[i] = sol
        else:
            x[i], y[i] = polar_to_cartesian(rs[i], thetas[i])
            
    return x, y

# 坐标转换函数polar_to_catersin
def polar_to_cartesian(r, theta):
    """将极坐标转换为直角坐标"""
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return x, y

In [140]:
# 计算待考虑碰撞龙头(与第二节端点)的x ,y坐标函数
def corner_point(x0, y0, x1, y1):
    """根据本块木板前后孔位的坐标计算本块木板的四个端点值"""
    orientation = atan2(y1-y0, x1-x0)
    fx = (x0-x1)/abs(x0-x1)
    fy = (y0-y0)/abs(y0-y1)
    mid_x0 = x0 + (27.5 / 100) * cos(orientation)*fx
    mid_y0 = y0 + (27.5 / 100) * sin(orientation)*fy
    mid_x1 = x1 - (27.5 / 100) * sin(orientation)*fx
    mid_y1 = y1 - (27.5 / 100) * sin(orientation)*fy
    left0_x = mid_x0 + 0.15 * cos(orientation - np.pi/2)
    left0_y = mid_y0 + 0.15 * sin(orientation - np.pi/2)
    right0_x = mid_x0 - 0.15 * cos(orientation - np.pi/2)
    right0_y = mid_y0 - 0.15 * sin(orientation - np.pi/2)
    left1_x = mid_x1 + 0.15 * cos(orientation - np.pi/2)
    left1_y = mid_y1 + 0.15 * sin(orientation - np.pi/2)
    right1_x = mid_x1 - 0.15 * cos(orientation - np.pi/2)
    right1_y = mid_y1 - 0.15 * sin(orientation - np.pi/2)
    ans = np.array([[left0_x, left0_y], [right0_x, right0_y], [left1_x, left1_y], [right1_x, right1_y]])
    return ans
    
    
# 计算点到直线距离的函数
def find_distance(x, y, x1, y1, x2, y2):
    """求点(坐标)到直线(由两点的坐标确定)的距离函数"""
    # 计算直线系数
    A = y2 - y1
    B = x1 - x2
    C = x2 * y1 - x1 * y2
    
    # 计算距离
    distance_point_to_line = abs(A * x + B * y + C) / np.sqrt(A**2 + B**2)
    
    return distance_point_to_line


In [146]:
# 计算等于0.15时对应的龙头theta值()
def g(vars, radius):
    """返回一个表示 min(点到直线距离)-0.15 的函数"""
    # 求龙身的位置
    x , y = vars
    xs = np.zeros(number)
    ys = np.zeros(number)
    xs, ys = body_state(x ,y , radius)
    
    # 求corner端点的位置
    points1 = corner_point(xs[0], ys[0], xs[1], ys[1])
    points2 = corner_point(xs[1], ys[1], xs[2], ys[2])
    points = np.vstack((points1, points2))
    
    # 求点到直线距离
    d = np.zeros((number-2)*8)
    j = 0
    for i in np.arange(3, number):
        for cp in range(8):
            d[j] = find_distance(points[cp][0], points[cp][1], xs[i], ys[i], xs[i+1], ys[i+1])*10 - 1.5
            j += 1
    min_distance = min(d)
    print('最小距离为',min_distance)
    if x**2 + y**2 > radius**2:
        equa1=(sqrt(x**2 + y**2) / b-atan2(y,x)) % (2*np.pi)  
    elif y>radius/3:
        equa1=abs(x**2 + (y-radius/3*2)**2 - (radius/3)**2)+abs(x-abs(x) )     
    else :
        equa1=abs(x**2 + (y+radius/3)**2 - (radius/3*2)**2)+abs(x+abs(x))   
    print(equa1) 
    return [equa1,min_distance]

for i in np.arange(0, 4.5, 0.1):
    radius_now = 4.5 - i
    sol, _, ier, msg = fsolve(g, [1., 1.], args = (radius_now, ), xtol=1e-8, full_output = True)
    print("半径为: ", radius_now,"时, 发生碰撞时龙头的位置: ", sol, "求解状态: ", ier)

/tmp/ipykernel_385384/941198728.py:20: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  sol = fsolve(eqinner, initial_guess, args = (x[i-1], y[i-1], distance[i-1], radius))
/tmp/ipykernel_385384/941198728.py:20: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last five Jacobian evaluations.
  sol = fsolve(eqinner, initial_guess, args = (x[i-1], y[i-1], distance[i-1], radius))


最小距离为 -0.4800407906018871
3.75
最小距离为 -0.4800407906018871
3.75
最小距离为 -0.4800407906018871
3.75
最小距离为 -0.4800426572397911
3.75
最小距离为 -0.4800395219504001
3.749999925494194
最小距离为 0.0
1.5802401899953256
最小距离为 -1.3368974757951226
1.7706748487166815
最小距离为 -0.4272164698657681
3.787776578047656
最小距离为 0.0
1.5802402575792502
最小距离为 0.0
1.5802401248027462
最小距离为 -0.26989218575990304
0.16306104274590183
最小距离为 -1.40889057954416
0.8853682419895126
最小距离为 -1.472082201222675
0.22178971020139082
最小距离为 -0.2698852124879436
0.1630610849392573
最小距离为 -0.2698867070039985
0.16306098318176732
最小距离为 -1.0936730027507129
0.00338523628009213
最小距离为 -0.7820146035292124
0.028423244340829434
最小距离为 -1.4981402200423382
0.09437913247740504
最小距离为 0.0
0.12837539393102881
最小距离为 0.0
0.05992492177260633
最小距离为 -0.9409370169681855
0.0012658155293352458
最小距离为 0.0
0.00023131197924719515
最小距离为 0.0
2.6793470033759093e-06
最小距离为 0.0
1.22622800802219e-10
最小距离为 0.0
4.440892098500626e-16
半径为:  4.5 时, 发生碰撞时龙头的位置:  [1.17544421 2.06815725] 求解状态

/tmp/ipykernel_385384/941198728.py:14: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  sol = fsolve(equations, initial_guess, args=(thetas[i-1], rs[i-1], distance[i-1]))


最小距离为 -0.9188853544447505
4.16949056549811 - pi
最小距离为 -0.797837352483431
2.06528061751089
最小距离为 -0.2717445385353645
0.40414581505087277
最小距离为 -0.2717448340080666
0.4041457244784632
最小距离为 -1.3923429275790384
0.016626649730558718
最小距离为 0.0
0.000980139858329121
最小距离为 0.0
4.574939600532346e-05
最小距离为 0.0
5.667034397305315e-09
最小距离为 -1.400035653557732
3.2862601528904634e-14
最小距离为 0.0
7.861578055212703e-11
半径为:  4.3 时, 发生碰撞时龙头的位置:  [0.91245569 1.76128481] 求解状态:  1
最小距离为 -1.2165428331716257
3.08
最小距离为 -1.2165428331716257
3.08
最小距离为 -1.2165428331716257
3.08
最小距离为 -1.2165003672499373
3.08
最小距离为 -1.2165410285817644
3.079999928474426
最小距离为 -0.8314243036945771
0.3287750374645708
最小距离为 -1.4470446966767314
0.15217702287704737
最小距离为 -1.2767880589954705
0.02717333888338369
最小距离为 -0.8304796381711307
0.32877506568853043
最小距离为 -0.8320415813630457
0.32877498079255796
最小距离为 -1.4827936005177682
0.3264398813584761


/tmp/ipykernel_385384/941198728.py:14: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last five Jacobian evaluations.
  sol = fsolve(equations, initial_guess, args=(thetas[i-1], rs[i-1], distance[i-1]))


最小距离为 -0.5176137269206105
0.32767807335984145
最小距离为 -1.4904752713328697
0.3254811134236473
最小距离为 -1.4823859598857305
0.32664155141679396
最小距离为 -0.5180467520827329
0.32767810160165967
最小距离为 -0.7190781294450181
0.32767801669839436
最小距离为 -0.7603054726592077
0.32730052607021554
最小距离为 -1.4756588091764025
0.3274834455435296
最小距离为 -1.475880213924232
0.3275762562025162
最小距离为 -0.7259489881930039
0.32763307401707875
最小距离为 -0.5145093001271388
0.3276685321207058
最小距离为 -1.3237230242780467
0.327680321049741
最小距离为 -0.4065641371324604
0.3276626376834706
最小距离为 -0.7027702480652483
0.3276567432553361
半径为:  4.2 时, 发生碰撞时龙头的位置:  [0.97345905 1.64239898] 求解状态:  1
最小距离为 0.0
2.87
最小距离为 0.0
2.87
最小距离为 0.0
2.87
最小距离为 0.0
2.87
最小距离为 0.0
2.869999929467837
最小距离为 -1.3181941396961363
6.19619122096947 - pi
最小距离为 -1.3631331135125615
0.43668329783899584
最小距离为 -0.6613667828700031
0.223959222107663
最小距离为 -1.0620059569781986
1.47398349607511
最小距离为 -1.0694318081466403
7.612016224330778
最小距离为 -0.6613897191326722
0.22395958768

In [147]:

# m,n = 7.63944916, 0.59382521
# (np.sqrt((m)**2 + n**2)/b - atan2(n, m))%(2*np.pi)

In [148]:
x=np.zeros(number)
y=np.zeros(number)
s=1.36092158 
k=2.3728361
x, y = body_state(s,k, 4.5)
print(x)
print(y)

[ 0.4943505  -0.91080389 -0.47638213 -1.48639466 -1.40545298  0.20400402
  1.44579934  1.70203896  2.7509591   1.70203896  0.13825203 11.49741722
  2.97359534  2.95918266  2.25080935  0.86156621  2.25080935  0.86156621
  2.25080935  0.86156621  2.25080935  0.86156621  2.25080935 -1.94003505
 -0.37281066  1.27593333  2.80274238  4.03021561  4.82469669  5.10696782
  4.8559083 ]
[ 2.320769    4.2538136   1.4936781   2.79842891  1.15041542  1.5139373
  2.60041987  0.97043789 -0.30324438  0.97043789  1.4968127   2.94333153
 -0.34325458 -1.99319163 -3.48339539 -4.37362205 -3.48339539 -4.37362205
 -3.48339539 -4.37362205 -3.48339539 -4.37362205 -3.48339539 -4.12229141
 -4.63834152 -4.57397349 -3.94839046 -2.84575296 -1.39962025  0.22605592
  1.85684381]


/tmp/ipykernel_385384/941198728.py:20: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last ten iterations.
  sol = fsolve(eqinner, initial_guess, args = (x[i-1], y[i-1], distance[i-1], radius))
/tmp/ipykernel_385384/941198728.py:20: RuntimeWarning: The iteration is not making good progress, as measured by the 
  improvement from the last five Jacobian evaluations.
  sol = fsolve(eqinner, initial_guess, args = (x[i-1], y[i-1], distance[i-1], radius))


In [149]:
m,n=1.36092158 ,2.37283619
sqrt(m**2+(n-4.5)**2)

2.52525908007147